In [4]:
import cv2
#Use the computer's default webcam.
cap = cv2.VideoCapture(0)

#ret tells us whether the frame was successfully captured.
# #Camera
#    ↓
# First picture
#    ↓
# old_frame
ret, old_frame = cap.read()
#Computer vision algorithms often work more simply with grayscale images.
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

# Find some points
#OpenCV looks at the image and finds interesting points, usually corners or places with strong changes.
#50 means we allow up to about 50 points.
#OpenCV uses the Shi–Tomasi method by default, and Shi–Tomasi uses:

# corner strength=min(λ1 and lamba2)
#0.3 quality level  for example 0.3*10
#7 minum distance
#"Keep points whose strength is at least 30% of the strongest point.
points = cv2.goodFeaturesToTrack(old_gray, 50, 0.3, 7)

while True:
    ret, frame = cap.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Track the points
    new_points, status, error = cv2.calcOpticalFlowPyrLK(
        old_gray, gray, points, None
    )

    for new, old in zip(new_points, points):
        x1, y1 = old.ravel()
        x2, y2 = new.ravel()

        # Draw movement arrow
        cv2.arrowedLine(
            frame,
            (int(x1), int(y1)),
            (int(x2), int(y2)),
            (0, 255, 0),
            2
        )

    cv2.imshow("Optical Flow", frame)

    # Press q to quit
    if cv2.waitKey(1) == ord('q'):
        break

    old_gray = gray
    points = new_points

cap.release()
cv2.destroyAllWindows()

#cv2.waitKey(1) waits 1 millisecond for a keyboard key press.
#ord('q') gives the numeric key code for q.

In [3]:
import cv2

# Open camera
cap = cv2.VideoCapture(0)

# Get first frame
ret, old_frame = cap.read()
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

while True:

    # Get current frame
    ret, frame = cap.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Find difference between frames
    difference = cv2.absdiff(old_gray, gray)

    # Convert difference to black and white
    _, motion = cv2.threshold(difference, 30, 255, cv2.THRESH_BINARY)

    # Show result
    cv2.imshow("Motion", motion)

    # Current frame becomes old frame
    old_gray = gray

    # Press q to quit
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [1]:
import cv2

cap = cv2.VideoCapture("C:/Users/HP/Downloads/person_walking_in_bedroom.mp4")

bg = cv2.createBackgroundSubtractorMOG2()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    mask = bg.apply(frame)

    cv2.imshow("Original", frame)
    cv2.imshow("Motion", mask)

    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [3]:
import cv2

# 1. Open the video (use 0 instead of the file name for a webcam)
cap = cv2.VideoCapture("C:/Users/HP/Downloads/person_walking_in_bedroom.mp4")

# 2. Create the MOG2 background subtractor
subtractor = cv2.createBackgroundSubtractorMOG2(history=200, varThreshold=40, detectShadows=True)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 3. Get the foreground mask (white = moving, grey = shadow, black = background)
    mask = subtractor.apply(frame)

    # 4. Keep only pure white pixels (removes shadows)
    _, mask = cv2.threshold(mask, 200, 255, cv2.THRESH_BINARY)

    # 5. Clean the mask: remove small noise dots, then fill small holes
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.dilate(mask, kernel, iterations=2)

    # 6. Find the contours (outlines of the white blobs)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 7. Draw a box around each big contour
    for c in contours:
        if cv2.contourArea(c) < 500:      # ignore tiny blobs
            continue
        x, y, w, h = cv2.boundingRect(c)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # 8. Show the result (press q to quit)
    cv2.imshow("Original with contours", frame)
    cv2.imshow("Mask", mask)
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [5]:

import cv2

# Open video
cap = cv2.VideoCapture("C:/Users/HP/Downloads/person_walking_in_bedroom.mp4")

# Create background subtractor
subtractor = cv2.createBackgroundSubtractorMOG2(
    history=200,
    varThreshold=40,
    detectShadows=True
)

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (5, 5)
)

while True:

    # Read frame
    ret, frame = cap.read()

    if not ret:
        break

    # Get foreground mask
    mask = subtractor.apply(frame)

    # Remove shadows
    _, mask = cv2.threshold(mask, 200, 255, cv2.THRESH_BINARY)

    # Remove noise
    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    # Make foreground regions bigger
    mask = cv2.dilate(
        mask,
        kernel,
        iterations=2
    )

    # Find contours
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for c in contours:

        # Ignore small objects/noise
        if cv2.contourArea(c) < 500:
            continue

        # Calculate moments
        M = cv2.moments(c)

        # Calculate centroid
        if M["m00"] != 0:

            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])

            # Draw centroid
            cv2.circle(
                frame,
                (cx, cy),
                7,
                (0, 0, 255),
                -1
            )
#cx + 10 means shift the text horizontally by 10 pixels to the right of the center point.
#Font size 0.6 and 2 is the thicknessof text.
            # Display coordinates
            cv2.putText(
                frame,
                f"Center: ({cx}, {cy})",
                (cx + 10, cy),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2
            )

            # Draw bounding box
            x, y, w, h = cv2.boundingRect(c)

            cv2.rectangle(
                frame,
                (x, y),
                (x + w, y + h),
                (0, 255, 0),
                2
            )

    # Show output
    cv2.imshow("Centroid Tracking", frame)
    cv2.imshow("Motion Mask", mask)

    # Press Q to quit
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [12]:
import cv2
import numpy as np

cap = cv2.VideoCapture(
    "C:/Users/HP/Downloads/two_people_room.mp4"
)

ret, prev = cap.read()

if not ret:
    print("Could not open video")
    exit()

prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)

# Create motion history image
mhi = np.zeros_like(prev_gray, dtype=np.float32)

# How long motion remains visible
duration = 1.0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Convert current frame to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Find difference between previous and current frame
    diff = cv2.absdiff(prev_gray, gray)

    # Threshold the difference
    _, motion = cv2.threshold(diff, 7, 255, cv2.THRESH_BINARY)

    # Current time in seconds
    #Computer timer → “How many tiny clock steps have passed?”
    #How many ticks happen in 1 second?
    #5,000,000 ticks = 5 seconds
    #1,000,000 ticks/second
    timestamp = cv2.getTickCount() / cv2.getTickFrequency()

    # Update Motion History Image
    mhi[motion > 0] = timestamp

    # Remove old motion
    mhi[timestamp - mhi > duration] = 0

    # Convert MHI to displayable image
    mhi_display = np.uint8(
        np.clip((mhi - (timestamp - duration)) / duration, 0, 1) * 255
    )

    # Show original video
    cv2.imshow("Original", frame)

    # Show motion history
    cv2.imshow("Motion History", mhi_display)

    # Current frame becomes previous frame
    prev_gray = gray.copy()

    # Press Q to quit
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [13]:
import cv2
import numpy as np

cap = cv2.VideoCapture(
    "C:/Users/HP/Downloads/person_walking_in_bedroom.mp4"
)

# Create Kalman Filter
kalman = cv2.KalmanFilter(4, 2)

# State = x, y, vx, vy
kalman.transitionMatrix = np.array([
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
], np.float32)

# We measure only x and y
kalman.measurementMatrix = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0]
], np.float32)

kalman.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03

kalman.measurementNoiseCov = np.eye(2, dtype=np.float32) * 0.5

# Background subtraction
subtractor = cv2.createBackgroundSubtractorMOG2()

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (5, 5)
)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Create foreground mask
    mask = subtractor.apply(frame)

    # Remove shadows/noise
    _, mask = cv2.threshold(
        mask,
        200,
        255,
        cv2.THRESH_BINARY
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    # Find contours
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # Kalman prediction
    prediction = kalman.predict()

    predicted_x = int(prediction[0])
    predicted_y = int(prediction[1])

    # Draw predicted position
    cv2.circle(
        frame,
        (predicted_x, predicted_y),
        8,
        (255, 0, 0),
        -1
    )

    # Find object
    for c in contours:

        if cv2.contourArea(c) < 500:
            continue

        # Get centroid
        M = cv2.moments(c)

        if M["m00"] != 0:

            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])

            # Measurement for Kalman Filter
            measurement = np.array(
                [[np.float32(cx)],
                 [np.float32(cy)]]
            )

            # Correct prediction using actual measurement
            kalman.correct(measurement)

            # Draw actual centroid
            cv2.circle(
                frame,
                (cx, cy),
                7,
                (0, 0, 255),
                -1
            )

            # Draw bounding box
            x, y, w, h = cv2.boundingRect(c)

            cv2.rectangle(
                frame,
                (x, y),
                (x + w, y + h),
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                "Actual",
                (cx + 10, cy),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2
            )

    cv2.putText(
        frame,
        "Blue = Prediction",
        (20, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 0, 0),
        2
    )

    cv2.putText(
        frame,
        "Red = Actual",
        (20, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255),
        2
    )

    cv2.imshow("Kalman Tracking", frame)
    cv2.imshow("Mask", mask)

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

C:\Users\HP\AppData\Local\Temp\ipykernel_2808\1565097311.py:71: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predicted_x = int(prediction[0])
C:\Users\HP\AppData\Local\Temp\ipykernel_2808\1565097311.py:72: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predicted_y = int(prediction[1])


In [16]:
import cv2

# Open video
cap = cv2.VideoCapture(
    "C:/Users/HP/Downloads/two_people_room.mp4"
)

# Read first frame
ret, frame = cap.read()

if not ret:
    print("Could not open video")
    exit()

# Select object to track
bbox = cv2.selectROI(
    "Select Object",
    frame,
    False
)

cv2.destroyWindow("Select Object")

# Create CSRT tracker
tracker = cv2.TrackerCSRT_create()

# Initialize tracker
tracker.init(frame, bbox)

while True:

    # Read next frame
    ret, frame = cap.read()

    if not ret:
        break

    # Track object
    success, bbox = tracker.update(frame)

    if success:

        # Convert bounding box values to integers
        x, y, w, h = map(int, bbox)

        # Draw tracking box
        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        # Display tracking status
        cv2.putText(
            frame,
            "Tracking",
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    else:

        cv2.putText(
            frame,
            "Tracking Lost",
            (50, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )

    cv2.imshow("CSRT Object Tracking", frame)

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [15]:
import cv2

# Open video
cap = cv2.VideoCapture(
    "C:/Users/HP/Downloads/person_walking_in_bedroom.mp4"
)

# Read first frame
ret, frame = cap.read()

if not ret:
    print("Could not open video")
    exit()

# Select object
bbox = cv2.selectROI(
    "Select Object",
    frame,
    False
)

cv2.destroyWindow("Select Object")

# Create KCF tracker
tracker = cv2.TrackerKCF_create()

# Initialize tracker
tracker.init(frame, bbox)

while True:

    # Read next frame
    ret, frame = cap.read()

    if not ret:
        break

    # Track object
    success, bbox = tracker.update(frame)

    if success:

        x, y, w, h = map(int, bbox)

        # Draw bounding box
        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            "Tracking",
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    else:

        cv2.putText(
            frame,
            "Tracking Lost",
            (50, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )

    cv2.imshow("KCF Object Tracking", frame)

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()